In [1]:
#!python.exe -m pip install --upgrade pip

In [2]:
#!pip install semantic-kernel[azure] 

In [3]:
# Make sure paths are correct for the imports

import os
import sys

notebook_dir = os.path.abspath("")
parent_dir = os.path.dirname(notebook_dir)
grandparent_dir = os.path.dirname(parent_dir)

sys.path.append(grandparent_dir)

In [4]:
import asyncio
from dotenv import load_dotenv

from semantic_kernel import Kernel
from semantic_kernel.contents import ChatHistory, ChatMessageContent, TextContent, ImageContent

# For AI connectors
from semantic_kernel.connectors.ai.open_ai import (
    AzureChatCompletion,
    OpenAIChatCompletion,
)

# For agents (if you're using Azure AI Agents features)
from semantic_kernel.agents import (
    AzureAIAgent,
    AzureAIAgentSettings,
    AzureAIAgentThread,
    ChatCompletionAgent,
    ChatHistoryAgentThread
)
from semantic_kernel.contents import (
    AnnotationContent,
    FunctionCallContent,
    FunctionResultContent,
)

# For Azure authentication (if using Azure services)
from azure.identity import DefaultAzureCredential
from azure.ai.agents.models import BingGroundingTool
from azure.ai.projects import AIProjectClient

In [5]:
kernel = Kernel()

In [6]:
from services import Service

from service_settings import ServiceSettings

service_settings = ServiceSettings()

# Select a service to use for this notebook (available services: OpenAI, AzureOpenAI, HuggingFace)
selectedService = (
    Service.AzureOpenAI
    if service_settings.global_llm_service is None
    else Service(service_settings.global_llm_service.lower())
)
print(f"Using service type: {selectedService}")



Using service type: Service.AzureOpenAI


In [7]:
# Remove all services so that this cell can be re-run without restarting the kernel
kernel.remove_all_services()

service_id = None
if selectedService == Service.OpenAI:
    from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion

    service_id = "default"
    kernel.add_service(
        OpenAIChatCompletion(
            service_id=service_id,
        ),
    )
    
elif selectedService == Service.AzureOpenAI:
    from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion

    service_id = "default"
    kernel.add_service(
        AzureChatCompletion(
            service_id=service_id,
        ),
    )

In [8]:
project = AzureAIAgent.create_client(
    credential=DefaultAzureCredential(),
    endpoint="https://resourceaifoundarytest.services.ai.azure.com/api/projects/firstProject")

# agent = project.agents.get_agent("asst_GLWunDDER6ajXJwxJ3cfnCUS")

# thread = project.agents.threads.get("thread_yp7jTMGN2IWbf3m7jGk14M4b")


In [9]:

bing_connection = await project.connections.get(name="maplehack")
conn_id = bing_connection.id
bing_grounding = BingGroundingTool(connection_id=conn_id)

In [10]:
# In cell 8a54a995
agent_definition = await project.agents.create_agent(
            name="BingGroundingAgent",
            instructions="Use the Bing grounding tool to answer the user's question. Add in Microsoft Build 2025 annoucments and figure out how that can help as well ",
            model="gpt-4.1",
            tools=bing_grounding.definitions,
        )

In [11]:
search_agent = AzureAIAgent(
            client=project,
            definition=agent_definition,
        )

In [12]:
core_comps_agent = ChatCompletionAgent(
    service=AzureChatCompletion(),
    name="CoreComps", # Renamed from CoreComponentsAgent
    instructions="Identify and explain the core Azure services that form the backbone of a requested data or AI pattern. Explain their individual roles and how they integrate."
)

cost_opt_agent = ChatCompletionAgent(
    service=AzureChatCompletion(),
    name="CostOpt", # Renamed from CostOptimizationAgent
    instructions="Outline methods to minimize Azure consumption costs for a given data or AI pattern. This includes considerations like resource sizing, pricing tiers, reservations, auto-scaling, and data lifecycle management."
)

perf_opt_agent = ChatCompletionAgent(
    service=AzureChatCompletion(),
    name="PerfOpt", # Renamed from PerformanceOptimizationAgent
    instructions="Detail techniques to maximize the speed and efficiency of a data or AI pattern. Address aspects such as data ingestion, processing, model inference, query performance, and caching."
)

azure_pattern_opt_triage_agent = ChatCompletionAgent(
    service=AzureChatCompletion(),
    name="agent", # Renamed from AzurePatternOptimizerTriageAgent
    instructions=(
        "You are an expert in Azure data and AI architecture and optimization. "
        "Your task is to evaluate user requests for specific Azure data or AI patterns and forward them to the appropriate specialized agents for targeted assistance. "
        "After gathering information from the specialized agents, provide the full, comprehensive answer to the user containing all relevant information."
        "ensure to put ALL the relivant MS documentation and provide links . prioritize Microsoft Build 2025 annoucments "
    ),
    plugins=[
        core_comps_agent,
        cost_opt_agent,
        perf_opt_agent,
        search_agent 
    ],
)

In [13]:
user_input = "what is the best azure pattern for data in databricks with 300 production on prem sql databases that are overloaded and moving that into the cloud" 

In [14]:
thread: ChatHistoryAgentThread = None


In [15]:
response = await azure_pattern_opt_triage_agent.get_response(
            messages=user_input,
            thread=thread,
        )

In [16]:
print(response.message.content)


Here’s a comprehensive strategy and the best Azure pattern for migrating 300 overloaded on-premises production SQL databases into Azure, making Databricks the central analytics platform. All patterns include up-to-date references, direct links to Microsoft documentation, and the latest relevant Microsoft Build 2025 announcements.

---

## 1. Recommended Azure Migration Pattern

### A. Assessment & Planning
1. **Discovery**: Use Azure Migrate and Data Migration Assistant to:
   - Inventory your SQL databases
   - Identify compatibility issues and dependencies
   - Segment by business criticality, workload, and size
   - References:
     - [Azure Migrate Documentation](https://learn.microsoft.com/en-us/azure/migrate/)
     - [Data Migration Assistant](https://learn.microsoft.com/en-us/sql/dma/dma-overview)

---

### B. Landing Zone Preparation
2. **Azure Enterprise-Scale Landing Zone**:
   - Prepare cloud networking, security, and identity for safe migration.
   - References:
     - [Azu